# SurGen CRC — Clean External Evaluation

Loads per-fold predictions, computes ZS + FT C-index + HR.
Saves results to `results/external_results/surgen.json`.

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
import warnings; warnings.filterwarnings('ignore')

from sparc.utils.external_eval import (
    load_perfold_slides, ensemble_slide, evaluate_cohort,
    print_results, save_results
)

In [ ]:
# ── Config ──
SQ_DIR  = Path('results/surgen_inference/mmp20_sq_cos20_knn64_es_20260303_132721_perfold')
IMG_DIR = Path('results/surgen_inference/mmp20_image_only_cos20_es_20260303_130951_perfold')
CLIN_CSV = Path('data/surgen_outcomes/SR386_labels.csv')
FIVE_YR = 5 * 365.25

MODELS = {
    'sq': 'mmp20_sq_cos20_knn64_es_20260303_132721',
    'img': 'mmp20_image_only_cos20_es_20260303_130951',
}

In [ ]:
# ── Load slides + clinical ──
sq_slides = load_perfold_slides(SQ_DIR)
img_slides = load_perfold_slides(IMG_DIR)
clin = pd.read_csv(CLIN_CSV)
print(f'Loaded {len(sq_slides)} SQ, {len(img_slides)} IMG slides')

rows = []
for _, r in clin.iterrows():
    cid = f"SR386_40X_HE_T{r['case_id']:03d}_01"
    if cid not in sq_slides or cid not in img_slides:
        continue
    if pd.isna(r['died_within_5_years']):
        continue

    died = int(r['died_within_5_years'])
    days = pd.to_numeric(r['days_till_death'], errors='coerce')
    T_os = days if (died and not np.isnan(days)) else FIVE_YR
    E_os = float(died)
    E_dss = 1.0 if (died and r['crc_primary_cause_of_death'] == 1) else 0.0
    site = str(r.get('site_of_tumour_grouping', '')).lower()
    ct = 'READ' if 'rect' in site else 'COAD'

    risk_sq, emb_sq = ensemble_slide(sq_slides[cid])
    risk_img, emb_img = ensemble_slide(img_slides[cid])

    rows.append({
        'id': cid, 'T_os': T_os, 'E_os': E_os, 'T_dss': T_os, 'E_dss': E_dss,
        'ct': ct, 'risk_sq': risk_sq, 'risk_img': risk_img,
        'emb_sq': emb_sq, 'emb_img': emb_img,
    })

df = pd.DataFrame(rows)
print(f'{len(df)} patients (COAD={sum(df.ct=="COAD")}, READ={sum(df.ct=="READ")})')

In [ ]:
# ── Evaluate ──
endpoints = [('DSS', 'T_dss', 'E_dss'), ('OS', 'T_os', 'E_os')]

res_coad = evaluate_cohort(df[df.ct == 'COAD'].reset_index(drop=True), endpoints, label='COAD')
res_read = evaluate_cohort(df[df.ct == 'READ'].reset_index(drop=True), endpoints, label='READ')

for r in [res_coad, res_read]:
    print_results(r)

save_results('surgen', [res_coad, res_read], MODELS,
             'results/external_results/surgen.json')